# CISM Tutorial 02: FANMOD+, Initialization, And Stringency Selection

This notebook starts from the output of tutorial 01.

By the end of this notebook you should have:

- a validated prepared dataset folder
- a loaded `CISM` object
- a `TissueStateDiscriminativeMotifs` object
- a discriminative-motifs-versus-stringency plot
- an AUC-based choice of stringency parameters
- a serialized artifact you can load in later analysis notebooks

## What This Notebook Covers

This notebook performs the transition from prepared graph txt files to an analysis-ready CISM state.

It will:

1. reuse the `network_dataset_root_path` and `dataset_folder` from tutorial 01
2. resolve the FANMOD+ binary already bundled in this repository
3. create tutorial-local `fanmod_output` and `fanmod_cache` folders
4. initialize `CISM`
5. load the prepared dataset
6. construct a discriminator
7. inspect discriminative motifs versus stringency
8. choose a stringency parameter using Random Forest ROC AUC
9. save a reusable pickle artifact for downstream notebooks

## Notes About FANMOD+ Binaries

This tutorial uses the FANMOD+ binaries already stored under `cism/FANMOD_binaries/`.

For tutorial purposes this is simpler and safer than moving them now, because the notebook can stay fully self-contained and directly point `CISM` to the repo-bundled executable.

In [ ]:
from pathlib import Path
import pickle
import platform

import pandas as pd

from cism import (
    CISM,
    DiscriminativeFeatureKey,
    HardDiscriminativeFC,
    TissueStateDiscriminativeMotifs,
    validate_network_dataset_directory,
)


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "tutorials" else Path.cwd().resolve()
TUTORIALS_DIR = PROJECT_ROOT / "tutorials"
TUTORIAL_RUNTIME_DIR = TUTORIALS_DIR / "runtime"
FANMOD_OUTPUT_ROOT = TUTORIAL_RUNTIME_DIR / "fanmod_output"
FANMOD_CACHE_ROOT = TUTORIAL_RUNTIME_DIR / "fanmod_cache"
SERIALIZED_ROOT = TUTORIAL_RUNTIME_DIR / "serialized"

FANMOD_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FANMOD_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
SERIALIZED_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Tutorial runtime directory: {TUTORIAL_RUNTIME_DIR}")
print(f"fanmod_output_root_path: {FANMOD_OUTPUT_ROOT}")
print(f"fanmod_cache_root_path: {FANMOD_CACHE_ROOT}")
print(f"serialized artifact directory: {SERIALIZED_ROOT}")


## Paste The Handoff Values From Tutorial 01

At the end of tutorial 01, the notebook printed these exact values:

- `network_dataset_root_path`
- `dataset_folder`

Use them directly here.

In [ ]:
# Replace these with the values printed at the end of tutorial 01.
network_dataset_root_path = str(PROJECT_ROOT / "data")
dataset_folder = "example_dataset"

dataset_dir = Path(network_dataset_root_path) / dataset_folder
print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")
print(f"dataset_dir = {dataset_dir}")


In [ ]:
# Confirm that tutorial 01 produced a valid folder.
txt_files = validate_network_dataset_directory(dataset_dir, require_patient_class=True)
print(f"Validated {len(txt_files)} txt graph files in {dataset_dir}")


## Resolve The FANMOD+ Binary

The `CISM` object expects:

- `fanmod_path`: directory containing the binary
- `fanmod_exe`: the executable filename

In [ ]:
fanmod_path = PROJECT_ROOT / "cism" / "FANMOD_binaries"

if platform.system().lower().startswith("win"):
    fanmod_exe = "LocalFANMOD - Windows"
else:
    fanmod_exe = "LocalFANMOD"

fanmod_binary_path = fanmod_path / fanmod_exe
print(f"fanmod_path = {fanmod_path}")
print(f"fanmod_exe = {fanmod_exe}")
print(f"fanmod binary exists: {fanmod_binary_path.exists()}")


## Initialize `CISM`

This tutorial uses the requested defaults:

- `motif_size = 3`
- `iterations = 1000`

In [ ]:
cism = CISM(
    fanmod_exe=fanmod_exe,
    fanmod_path=str(fanmod_path),
    network_dataset_root_path=network_dataset_root_path,
    fanmod_output_root_path=str(FANMOD_OUTPUT_ROOT),
    fanmod_cache_root_path=str(FANMOD_CACHE_ROOT),
    motif_size=3,
    iterations=1000,
)

cism


## Load The Prepared Dataset

Use the requested defaults when adding the dataset:

- `force_run_fanmod=False`
- `force_parse=False`
- `n_jobs=12`

In [ ]:
# Replace dataset_type and dataset_name with the metadata you want in downstream analysis.
dataset_type = "Disease"
dataset_name = "Example"

cism.add_dataset(
    dataset_folder=dataset_folder,
    dataset_type=dataset_type,
    dataset_name=dataset_name,
    force_run_fanmod=False,
    force_parse=False,
    n_jobs=12,
    require_patient_class=True,
)


In [ ]:
motif_df = cism.motif_dataset()
print(f"Loaded motif rows: {0 if motif_df is None else len(motif_df)}")
display(motif_df.head() if motif_df is not None else None)


## Define The Analysis Metadata Needed For Discrimination

To analyze discriminative motifs, we need:

- a path to `patient_class.csv`
- a mapping from raw class values to readable labels
- a mapping from integer cell-type ids to cell-type names

Both mappings are dataset-specific, so replace the examples below with your real values.

In [ ]:
tissue_state_csv_path = str(dataset_dir / "patient_class.csv")

# Example: raw labels in patient_class.csv -> readable class names.
tissue_state_to_string = {
    "POSITIVE": "POSITIVE",
    "NEGATIVE": "NEGATIVE",
}

# Replace this with your real integer-id -> cell-type-name mapping.
common_cells_type = {
    0: "Tumor",
    1: "TCell",
}

# These are the two classes we will compare in this tutorial.
labels = ["POSITIVE", "NEGATIVE"]


In [ ]:
discriminator = TissueStateDiscriminativeMotifs(
    cism=cism,
    tissue_state_csv_path=tissue_state_csv_path,
    tissue_state_to_string=tissue_state_to_string,
    common_cells_type=common_cells_type,
)

discriminator


## Plot Discriminative Motifs Per Stringency

This gives a first view of how many motifs remain as the discrimination stringency parameter increases.

Replace the colors if you prefer a different class palette.

In [ ]:
class_to_color = {
    labels[0]: "tab:red",
    labels[1]: "tab:blue",
}

discover_result = discriminator.discover(
    extract_by=DiscriminativeFeatureKey.STRUCTURE_AND_CELL_IDENTITIES,
    classes=labels,
)

discover_result.plot_number_of_motifs_versus_discrimination_stringency_parameter(
    class_to_color=class_to_color,
)


## Define A Feature Configuration Template

We will tune the main stringency controls used by the Random Forest validation:

- `shared_percentage`
- `max_class_features`

The configuration below is only the starting template. Optuna will search around it.

In [ ]:
feature_conf_template = HardDiscriminativeFC(
    labels=labels,
    extract_by=DiscriminativeFeatureKey.STRUCTURE_AND_CELL_IDENTITIES,
    use_cells_type_composition=False,
    use_motifs=True,
    shared_percentage=0.5,
    max_class_features=10,
)

feature_conf_template


## Choose The Best Stringency By Random Forest ROC AUC

This step uses the built-in Optuna tuner to search for a good stringency setting.

The objective is `roc_auc`, which is computed from the existing CISM motif-analysis pipeline.

In [ ]:
tuning_result = discriminator.tune_stringency(
    feature_conf=feature_conf_template,
    n_trials=20,
    metric="roc_auc",
    random_state=0,
    n_jobs=12,
    prefer="processes",
    shared_percentage_range=(0.1, 0.95),
    max_class_features_range=(1, 25),
)

print("Best parameters:")
print(tuning_result.best_params)
print(f"Best ROC AUC: {tuning_result.best_score:.4f}")


In [ ]:
tuning_trials_df = tuning_result.trials_dataframe()
display(tuning_trials_df)


## Build The Selected Analysis Configuration

We now convert the tuned parameters into the feature configuration we will carry into later notebooks.

In [ ]:
best_feature_conf = HardDiscriminativeFC(
    labels=labels,
    extract_by=DiscriminativeFeatureKey.STRUCTURE_AND_CELL_IDENTITIES,
    use_cells_type_composition=False,
    use_motifs=True,
    shared_percentage=tuning_result.best_params["shared_percentage"],
    max_class_features=tuning_result.best_params["max_class_features"],
)

best_feature_conf


## Optional: Run One Final Analysis Pass With The Selected Stringency

This is not strictly required for moving forward, but it gives you a first concrete analysis object at the selected setting.

In [ ]:
best_analysis_result = discriminator.analyze_motifs(
    feature_conf=best_feature_conf,
    exclude_patients=[],
    n_jobs=12,
    prefer="processes",
)

print(f"Selected-setting ROC AUC: {best_analysis_result.get_roc_auc_score():.4f}")
display(best_analysis_result.results.head())


## Save A Reusable Analysis-Ready Artifact

The main goal is to leave this notebook with something you can load directly in later analysis notebooks.

We save:

- the `CISM` object
- the discriminator
- the selected feature configuration
- the tuning summary table
- the key path/config metadata

In [ ]:
artifact_path = SERIALIZED_ROOT / f"{dataset_folder}_cism_ready.pkl"

artifact = {
    "cism": cism,
    "discriminator": discriminator,
    "best_feature_conf": best_feature_conf,
    "tuning_best_params": tuning_result.best_params,
    "tuning_best_score": tuning_result.best_score,
    "tuning_trials_df": tuning_trials_df,
    "network_dataset_root_path": network_dataset_root_path,
    "dataset_folder": dataset_folder,
    "fanmod_path": str(fanmod_path),
    "fanmod_exe": fanmod_exe,
    "fanmod_output_root_path": str(FANMOD_OUTPUT_ROOT),
    "fanmod_cache_root_path": str(FANMOD_CACHE_ROOT),
    "tissue_state_csv_path": tissue_state_csv_path,
    "tissue_state_to_string": tissue_state_to_string,
    "common_cells_type": common_cells_type,
    "labels": labels,
}

with open(artifact_path, "wb") as handle:
    pickle.dump(artifact, handle)

print(f"Saved analysis-ready artifact: {artifact_path}")


In [ ]:
print("Notebook 02 summary:")
print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")
print(f"fanmod_path = {fanmod_path}")
print(f"fanmod_exe = {fanmod_exe}")
print(f"fanmod_output_root_path = {FANMOD_OUTPUT_ROOT}")
print(f"fanmod_cache_root_path = {FANMOD_CACHE_ROOT}")
print(f"best stringency shared_percentage = {best_feature_conf.shared_percentage}")
print(f"best stringency max_class_features = {best_feature_conf.max_class_features}")
print(f"artifact_path = {artifact_path}")


## You Are Done When...

Before moving to the analysis notebook, confirm that:

- the dataset from tutorial 01 validated successfully
- FANMOD+ was resolved correctly from `cism/FANMOD_binaries/`
- `CISM(...)` was initialized with `motif_size=3` and `iterations=1000`
- `cism.add_dataset(...)` completed with:
  - `force_run_fanmod=False`
  - `force_parse=False`
  - `n_jobs=12`
- the discriminative-motifs-versus-stringency plot was generated
- the best stringency was chosen from Random Forest ROC AUC
- the pickle artifact was saved successfully

At that point, you have something analysis-ready that later notebooks can load directly.